In [1]:
import torch
from transformers import AutoTokenizer,AutoModel

In [2]:
sentences=[
    "i took my dog for a walk",
    "i took my cat for a walk",
    "Today is going to rain",
]

In [3]:
model_ckpt="sentence-transformers/all-MiniLM-L6-v2"
tokenizer=AutoTokenizer.from_pretrained(model_ckpt)
model=AutoModel.from_pretrained(model_ckpt)

encoded_input=tokenizer(sentences,padding=True,truncation=True,return_tensors="pt")

with torch.no_grad():
    model_output=model(**encoded_input)

token_embeddings=model_output.last_hidden_state
print(token_embeddings.shape)

Loading weights:   0%|          | 0/103 [00:00<?, ?it/s]

torch.Size([3, 9, 384])


In [4]:
import torch
import torch.nn.functional as F

def mean_pooling(model_output, attention_mask):
    token_embeddings = model_output[0] #First element of model_output contains all token embeddings
    input_mask_expanded = attention_mask.unsqueeze(-1).expand(token_embeddings.size()).float()
    return torch.sum(token_embeddings * input_mask_expanded, 1) / torch.clamp(input_mask_expanded.sum(1), min=1e-9)


sentence_embeddings = mean_pooling(model_output, encoded_input['attention_mask'])

sentence_embedding=F.normalize(sentence_embeddings,p=2,dim=1)
print("Sentence embeddings:")
print(sentence_embedding.size())

Sentence embeddings:
torch.Size([3, 384])


In [5]:
from datasets import load_dataset

squad =load_dataset("rajpurkar/squad",split="validation[:100]")
squad

Dataset({
    features: ['id', 'title', 'context', 'question', 'answers'],
    num_rows: 100
})

In [6]:
def get_embeddings(text_list):
  device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
  encoded_input= tokenizer(text_list,padding=True,truncation=True,return_tensors="pt")
  encoded_input={k:v.to(device) for k,v in encoded_input.items()}
  with torch.no_grad():
    model_output=model(**encoded_input)
  return mean_pooling(model_output,encoded_input['attention_mask'])

squad_with_embeddings= squad.map(lambda x:{"embeddings": get_embeddings(x["context"]).cpu().numpy()[0]})

In [7]:
import torch
print("Torch:", torch.__version__)

!pip show torchvision

Torch: 2.14.0+cu130


In [9]:
!pip uninstall -y torchvision

In [11]:
squad_with_embeddings.add_faiss_index(column="embeddings")

question="Who headlined the halftime show for Super Bowl 50?"
question_embedding=get_embeddings([question]).cpu().detach().numpy()
question_embedding.shape
scores,samples= squad_with_embeddings.get_nearest_examples(
    "embeddings",question_embedding,k=10
)


  0%|          | 0/1 [00:00<?, ?it/s]

In [12]:
for i in range(10):
    print(f"\n--- Result {i+1} ---")
    print("Score:", scores[i])
    print("Context:", samples["context"][i])


--- Result 1 ---
Score: 23.663609
Context: CBS broadcast Super Bowl 50 in the U.S., and charged an average of $5 million for a 30-second commercial during the game. The Super Bowl 50 halftime show was headlined by the British rock group Coldplay with special guest performers Beyoncé and Bruno Mars, who headlined the Super Bowl XLVII and Super Bowl XLVIII halftime shows, respectively. It was the third-most watched U.S. broadcast ever.

--- Result 2 ---
Score: 23.663609
Context: CBS broadcast Super Bowl 50 in the U.S., and charged an average of $5 million for a 30-second commercial during the game. The Super Bowl 50 halftime show was headlined by the British rock group Coldplay with special guest performers Beyoncé and Bruno Mars, who headlined the Super Bowl XLVII and Super Bowl XLVIII halftime shows, respectively. It was the third-most watched U.S. broadcast ever.

--- Result 3 ---
Score: 23.663609
Context: CBS broadcast Super Bowl 50 in the U.S., and charged an average of $5 million 